In [4]:
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
# Q1. 고객별 전환율을 계산하고, 전환율이 가장 높은 상위 10명 고객의 전환율을 계산

In [5]:
df = pd.read_csv(r"C:\Users\kicki\제로베이스파이썬\Excel_Practice\인터넷쇼핑몰로그.csv")

In [6]:
df.head() # 구매하지않았다면 결측(NaN)으로 뜬다

,customer_id,session_id,timestamp,event_type,amount
0,C0001,ZH6JLCVUE5W,2020-01-01 00:00:00,search,NaN
1,C0001,ZH6JLCVUE5W,2020-01-01 00:00:13,view,NaN
2,C0001,44O99BT83Y4T,2020-01-21 03:04:09,search,NaN
3,C0001,44O99BT83Y4T,2020-01-21 03:07:06,view,NaN
4,C0001,44O99BT83Y4T,2020-01-21 03:10:02,add_to_cart,NaN


In [7]:
# 고객이 몇번을 들어왔는지 확인 # .apply(Set)을 사용하면 중복이 없어진다 
num_visit = df.groupby('customer_id')['session_id'].apply(set).apply(len) # apply(len)는 왜? 집합이닌깐 집합의 길이를 구하면 Session_id 숫자 카운트가능
num_visit
# 고객별 전환률은 구매count / num_visit로 계산

customer_id
C0001    47
C0002     8
C0003    19
C0004    51
C0005    61
         ..
C0996    55
C0997    10
C0998    63
C0999    74
C1000    16
Name: session_id, Length: 1000, dtype: int64

In [8]:
num_purchase = df.loc[df['event_type'] == 'purchase'].groupby('customer_id')['session_id'].apply(set).apply(len)
num_purchase

customer_id
C0001    34
C0002     3
C0003     5
C0004    22
C0005    19
         ..
C0996    20
C0997     2
C0998    16
C0999    32
C1000    10
Name: session_id, Length: 993, dtype: int64

In [9]:
# 고객 방문과 구매 두 2개 series를 묶는다 + 결측제거
result = pd.concat([num_visit, num_purchase], axis = 1) # 여기에서는 ''가 안 붙고
result.columns = ['num_visit', 'num_purchase'] # 여기서는 붙는다....ㅜㅜ
result['num_visit']= result['num_visit'].fillna(0).astype(int) # 빈칸이 있으면 0으로 변환
result['conversion_rate'] = result['num_purchase'] / result['num_visit'] * 100
result.sort_values(by = 'conversion_rate', ascending = False).head(10)

# 쉬운버젼
# (num_purchase) / (num_visit).fillna().sort_value(ascending = False) * 100.iloc[:10]

,num_visit,num_purchase,conversion_rate
customer_id,,,
C0539,36,35.0,97.222222
C0885,28,27.0,96.428571
C0582,20,19.0,95.000000
C0246,50,47.0,94.000000
C0777,50,46.0,92.000000
C0454,74,68.0,91.891892
C0234,48,44.0,91.666667
C0568,67,61.0,91.044776
C0798,55,50.0,90.909091


In [10]:
#Q2. 고객별 평균 방문 주기, 구매 총 금액, 마지막 방문 날짜를 추출하세요. 
     #단 평균 방문주기와 마지막 방문주기는 세션 시작 시각을 기준으로 측정합니다.b

In [11]:
df.head()

,customer_id,session_id,timestamp,event_type,amount
0,C0001,ZH6JLCVUE5W,2020-01-01 00:00:00,search,NaN
1,C0001,ZH6JLCVUE5W,2020-01-01 00:00:13,view,NaN
2,C0001,44O99BT83Y4T,2020-01-21 03:04:09,search,NaN
3,C0001,44O99BT83Y4T,2020-01-21 03:07:06,view,NaN
4,C0001,44O99BT83Y4T,2020-01-21 03:10:02,add_to_cart,NaN


In [12]:
# 평균 방문 주기 - timestamp (min)간의 간격을 바탕으로 평균산출
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [13]:
session_start_time_df = df.groupby(['customer_id', 'session_id'])['timestamp'].min().reset_index()
session_start_time_df = session_start_time_df.sort_values(by = ['customer_id', 'timestamp'])
session_start_time_df.head()

,customer_id,session_id,timestamp
46,C0001,ZH6JLCVUE5W,2020-01-01 00:00:00
6,C0001,44O99BT83Y4T,2020-01-21 03:04:09
22,C0001,EGLJ7AG490Y8,2020-02-19 04:51:03
3,C0001,1RCCJ795,2020-03-21 18:41:21
29,C0001,IG8PLDN9,2020-04-15 22:26:53


In [14]:
visit_freq = []
for customer_id in session_start_time_df['customer_id'].unique(): # 왜 Uique를 붙여줘야되??...ㅜㅜ
    customer_df = df.loc[df['customer_id'] == customer_id]
    visit_freq.append(customer_df['timestamp'].diff().dt.days.mean())

visit_freq = pd.Series(visit_freq, index = session_start_time_df['customer_id'].unique())

In [15]:
visit_freq.head()

C0001     6.522222
C0002     8.136364
C0003    12.375000
C0004     8.866242
C0005    10.780488
dtype: float64

In [16]:
# 구매 총 금액
df.loc[df['amount'].notnull(), 'event_type'].unique() # .notnull()은 결측이 아닌것을 의미. Amount가 결측 아닌것중 event_type을 unique value로.

array(['purchase'], dtype=object)

In [17]:
total_amount = df.groupby('customer_id')['amount'].sum() # 여기서 썸은 Python 에서 결측을 자동으로 빼주고 계산해준다
total_amount.sort_values(ascending = False).head()

customer_id
C0236    63399.0
C0235    57973.0
C0130    56695.0
C0847    54398.0
C0694    54150.0
Name: amount, dtype: float64

In [18]:
# 마지막 방문 날짜; 위의 조건이 세션 시작 시각을 기준으로 측정한다고 했으니, 중간단계를 거쳐야 한다. session_id 뽑고, 거기서 가장 작은 시간 추출

#1
last_sessions = df.loc[df.groupby('customer_id')['timestamp'].idxmax(), 'session_id'] #idxmax는 timestamp max 값 가져오라는 의미
last_sessions.head()


180      A4WN8O50
203      YJBAQHHA
244    CFT6DY87ZK
402    X1BC4L5S4W
567      YWY1EMDT
Name: session_id, dtype: object

In [19]:
#2
last_visit_time = df.loc[df['session_id'].isin(last_sessions)].groupby('customer_id')['timestamp'].min()
last_visit_time.head()

customer_id
C0001   2023-04-08 19:45:16
C0002   2020-07-01 19:18:21
C0003   2021-05-19 01:26:31
C0004   2023-11-16 21:00:17
C0005   2024-11-28 00:33:39
Name: timestamp, dtype: datetime64[ns]

In [20]:
# 지금까지 결과 도출한 고객명별로 3개를 합쳐준다 visit freq, total amount, last visit time
result = pd.concat([visit_freq, total_amount, last_visit_time], axis = 1)
result.columns = ['평균방문주기', '총구매금액', '마지막방문일'] # 칼럼 정의해준다
result.head()

,평균방문주기,총구매금액,마지막방문일
C0001,6.522222,20920.0,2023-04-08 19:45:16
C0002,8.136364,1665.0,2020-07-01 19:18:21
C0003,12.375000,5908.0,2021-05-19 01:26:31
C0004,8.866242,17239.0,2023-11-16 21:00:17
C0005,10.780488,16845.0,2024-11-28 00:33:39


#Q3. 결제창 (checkout) 까지 갔으나 구매 (purchase)를 하지않고 이탈한 세션의 비율을 계산한다.

In [23]:
checkout = set(df.loc[df['event_type'] == 'checkout', 'session_id'].unique()) # 왜 Set으로 묶어줬는지는 하기 계산(len)을 하기위해서
purchase = set(df.loc[df['event_type'] == 'purchase', 'session_id'].unique())

In [28]:
(1 - len(checkout & purchase) / len(checkout)) * 100 # 분자가 교집한 이유는 물론 checkout후 purchase 이지만, make sure하기위해.

5.348807331591798

In [ ]:
Q4. 가장 흔히 발생하는 고객여정 (시퀀스) 을 세개까지 추출하시오

In [93]:
# 이벤트 단위로 묶어본다
event_sequence = df.groupby('session_id')['event_type'].apply(lambda x:'-'.join(x)) # apply는 여기서 형식을 바꿔줄때 쓴다. 즉 무엇인가를 변형할때

In [94]:
event_sequence

session_id
0003WO0OT                                     search-view
000EP3IYW5Y2     search-view-add_to_cart-remove_from_cart
001P4TGFE                                     search-view
002G7L43MBC              search-view-add_to_cart-checkout
002XVE74O                                            view
                                  ...                    
ZZW2MTJPJ                                            view
ZZX615TMQ1F     search-view-add_to_cart-checkout-purchase
ZZY33AQKWQKT    search-view-add_to_cart-checkout-purchase
ZZY6N9U8QXOT    search-view-add_to_cart-checkout-purchase
ZZZ5TJOGIIV             view-add_to_cart-remove_from_cart
Name: event_type, Length: 48461, dtype: object

In [96]:
event_sequence.value_counts(normalize = True).iloc[:3] # top 3개 조합을 구하는것
# 하기 정보에 의하면 검색 ~ 구매까지 이어진 비율이 가장 높다 >> 굿

event_type
search-view-add_to_cart-checkout-purchase    0.229009
view-add_to_cart-checkout-purchase           0.227069
search-view                                  0.201585
Name: proportion, dtype: float64